# Financial Stress Prediction - Exploratory Data Analysis

This notebook performs exploratory analysis on the synthetic transaction data.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Libraries imported successfully")

## 1. Load Data

In [ ]:
# Load transaction data
transactions = pd.read_csv('../data/processed/all_transactions.csv')
transactions['transaction_date'] = pd.to_datetime(transactions['transaction_date'])

# Load engineered features
features = pd.read_csv('../data/processed/engineered_features.csv')

print(f"Transactions shape: {transactions.shape}")
print(f"Features shape: {features.shape}")
print(f"\nTransaction columns: {list(transactions.columns)}")
print(f"\nFeature columns: {list(features.columns)}")

## 2. Target Distribution

In [ ]:
# Target distribution
target_dist = features['is_stressed'].value_counts()

fig = px.pie(
    values=target_dist.values,
    names=['Not Stressed', 'Stressed'],
    title='Distribution of Financial Stress',
    color_discrete_sequence=['#2ecc71', '#e74c3c']
)
fig.show()

print(f"\nTarget distribution:")
print(f"Not Stressed: {target_dist[0]} ({target_dist[0]/len(features)*100:.1f}%)")
print(f"Stressed: {target_dist[1]} ({target_dist[1]/len(features)*100:.1f}%)")

## 3. Transaction Type Analysis

In [ ]:
# Transaction type distribution
txn_type_dist = transactions['transaction_type'].value_counts()

fig = px.bar(
    x=txn_type_dist.index,
    y=txn_type_dist.values,
    title='Transaction Type Distribution',
    labels={'x': 'Transaction Type', 'y': 'Count'}
)
fig.show()

# Transaction volume by customer stress status
stressed_customers = features[features['is_stressed'] == 1]['customer_id'].unique()
transactions['is_stressed'] = transactions['customer_id'].isin(stressed_customers)

txn_by_stress = transactions.groupby(['transaction_type', 'is_stressed']).size().unstack()

fig = px.bar(
    txn_by_stress,
    barmode='group',
    title='Transaction Types: Stressed vs Not Stressed Customers'
)
fig.show()

## 4. Salary Pattern Analysis

In [ ]:
# Days since last salary
fig = px.histogram(
    features,
    x='days_since_last_salary',
    color='is_stressed',
    nbins=30,
    title='Days Since Last Salary Distribution',
    labels={'is_stressed': 'Stressed'},
    barmode='overlay',
    opacity=0.7
)
fig.show()

# Salary delay trend
fig = px.box(
    features,
    x='is_stressed',
    y='salary_delay_trend',
    title='Salary Delay Trend: Stressed vs Not Stressed',
    labels={'is_stressed': 'Stressed', 'salary_delay_trend': 'Salary Delay Trend'}
)
fig.show()

## 5. Balance Analysis

In [ ]:
# Balance drop percentage
fig = px.box(
    features,
    x='is_stressed',
    y='balance_drop_pct_4weeks',
    title='4-Week Balance Drop: Stressed vs Not Stressed',
    labels={'is_stressed': 'Stressed', 'balance_drop_pct_4weeks': 'Balance Drop %'}
)
fig.show()

# Current balance distribution
fig = px.histogram(
    features,
    x='current_balance',
    color='is_stressed',
    nbins=50,
    title='Current Balance Distribution',
    labels={'is_stressed': 'Stressed'},
    barmode='overlay',
    opacity=0.7
)
fig.show()

## 6. UPI to Loan Apps Analysis

In [ ]:
# UPI to loan apps percentage
fig = px.box(
    features,
    x='is_stressed',
    y='upi_to_loan_apps_pct',
    title='UPI to Loan Apps %: Stressed vs Not Stressed',
    labels={'is_stressed': 'Stressed', 'upi_to_loan_apps_pct': 'UPI to Loan Apps %'}
)
fig.show()

# Loan app transaction amount
fig = px.scatter(
    features,
    x='upi_loan_amount_30d',
    y='balance_drop_pct_4weeks',
    color='is_stressed',
    title='Loan App UPI Amount vs Balance Drop',
    labels={
        'upi_loan_amount_30d': 'UPI to Loan Apps (30d)',
        'balance_drop_pct_4weeks': 'Balance Drop %',
        'is_stressed': 'Stressed'
    }
)
fig.show()

## 7. Spending Pattern Analysis

In [ ]:
# Category spend ratios
spend_cols = ['essential_spend_ratio', 'discretionary_spend_ratio', 'cash_withdrawal_ratio']

for col in spend_cols:
    fig = px.box(
        features,
        x='is_stressed',
        y=col,
        title=f'{col.replace("_", " ").title()}: Stressed vs Not Stressed',
        labels={'is_stressed': 'Stressed'}
    )
    fig.show()

## 8. Payment Behavior Analysis

In [ ]:
# Bill payment day
fig = px.histogram(
    features,
    x='avg_bill_payment_day',
    color='is_stressed',
    nbins=30,
    title='Average Bill Payment Day Distribution',
    labels={'is_stressed': 'Stressed'},
    barmode='overlay',
    opacity=0.7
)
fig.show()

# Failed autopay
fig = px.histogram(
    features,
    x='failed_autopay_count',
    color='is_stressed',
    nbins=20,
    title='Failed Autopay Count Distribution',
    labels={'is_stressed': 'Stressed'},
    barmode='overlay',
    opacity=0.7
)
fig.show()

## 9. Correlation Analysis

In [ ]:
# Select numeric columns
numeric_cols = features.select_dtypes(include=[np.number]).columns
numeric_cols = [col for col in numeric_cols if col not in ['customer_id', 'is_stressed']]

# Calculate correlations with target
correlations = features[numeric_cols + ['is_stressed']].corr()['is_stressed'].sort_values(ascending=False)

# Plot top correlations
top_features = correlations.abs().sort_values(ascending=False)[1:16]  # Top 15

fig = px.bar(
    x=top_features.values,
    y=top_features.index,
    orientation='h',
    title='Top 15 Features Correlated with Financial Stress',
    labels={'x': 'Correlation', 'y': 'Feature'}
)
fig.show()

print("\nTop 10 features correlated with stress:")
print(correlations[1:11])

## 10. Key Insights Summary

**High-Risk Indicators:**
1. Increasing salary payment delays
2. Significant balance drops over 4 weeks
3. High percentage of UPI to loan apps
4. Delayed bill payments
5. Reduced discretionary spending
6. Increased ATM withdrawals

**Next Steps:**
- Use these insights to train predictive models
- Focus on features with high correlation to stress
- Consider feature interactions